In [2]:
from google.colab import files
uploaded = files.upload()

Saving cs-training.csv to cs-training.csv


In [3]:
import pandas as pd
df = pd.read_csv("cs-training.csv")

df.drop(columns=["Unnamed: 0"], inplace=True, errors='ignore')

In [4]:
df.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [5]:
print(df.shape)
print(df.head())
print(df.info())

(150000, 11)
   SeriousDlqin2yrs  RevolvingUtilizationOfUnsecuredLines  age  \
0                 1                              0.766127   45   
1                 0                              0.957151   40   
2                 0                              0.658180   38   
3                 0                              0.233810   30   
4                 0                              0.907239   49   

   NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  MonthlyIncome  \
0                                     2   0.802982         9120.0   
1                                     0   0.121876         2600.0   
2                                     1   0.085113         3042.0   
3                                     0   0.036050         3300.0   
4                                     1   0.024926        63588.0   

   NumberOfOpenCreditLinesAndLoans  NumberOfTimes90DaysLate  \
0                               13                        0   
1                                4               

In [6]:
print(df.columns)

Index(['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age',
       'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome',
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate',
       'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse',
       'NumberOfDependents'],
      dtype='object')


In [7]:
print(df.isnull().sum())

SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64


In [9]:
# 1. Create indicator for high-missing feature
df["MonthlyIncome_missing"] = df["MonthlyIncome"].isnull().astype(int)

# 2. Impute values
df.fillna({
    "MonthlyIncome": df["MonthlyIncome"].median(),
    "NumberOfDependents": df["NumberOfDependents"].median()
}, inplace=True)

In [11]:
print(df["SeriousDlqin2yrs"].value_counts())
print(df["SeriousDlqin2yrs"].value_counts(normalize=True))

SeriousDlqin2yrs
0    139974
1     10026
Name: count, dtype: int64
SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64


In [12]:
from sklearn.model_selection import train_test_split

X = df.drop("SeriousDlqin2yrs", axis=1)
y = df["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [13]:
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
print(scale_pos_weight)

13.960728088766986


In [14]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [15]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

In [16]:
import numpy as np

np.save("y_test.npy", y_test)
np.save("y_pred.npy", y_pred)
np.save("y_prob.npy", y_prob)

In [17]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob[:,1]))
print(classification_report(y_test, y_pred))

Accuracy: 0.8055
ROC-AUC: 0.8677241349207797
              precision    recall  f1-score   support

           0       0.98      0.81      0.89     27995
           1       0.22      0.77      0.34      2005

    accuracy                           0.81     30000
   macro avg       0.60      0.79      0.62     30000
weighted avg       0.93      0.81      0.85     30000



In [18]:
import joblib
joblib.dump(model, "xgb_model.pkl")

['xgb_model.pkl']

In [19]:
np.save("X_test.npy", X_test)

In [21]:
import os

os.makedirs("models", exist_ok=True)
os.makedirs("experiments", exist_ok=True)

In [22]:
joblib.dump(model, "models/xgb_model.pkl")
np.save("experiments/X_test.npy", X_test)
np.save("experiments/y_test.npy", y_test)
np.save("experiments/y_pred.npy", y_pred)
np.save("experiments/y_prob.npy", y_prob)

In [23]:
import os
print(os.listdir())

['.config', 'y_prob.npy', 'X_test.npy', 'experiments', 'y_test.npy', 'xgb_model.pkl', 'cs-training.csv', 'models', 'y_pred.npy', 'sample_data']
